# Détection d'intrusions — Log Cloud
## Projet IDS | Apprentissage non supervisé — Isolation Forest

In [1]:
import pandas as pd
import numpy as np
import re
import os

os.chdir(r'C:\Users\asus\Desktop\IDS')

# ========== CHARGEMENT AVEC LE BON CHEMIN ==========
# Ton fichier est dans le dossier Data/
file_path = 'Non supervisé/Data/logs cloud.csv'  # Chemin relatif

# Vérifier si le fichier existe
if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"✅ Fichier chargé: {file_path}")
    print(f"Shape initial: {df.shape}")
else:
    print(f"❌ Fichier non trouvé: {file_path}")
    print("Vérifie les fichiers disponibles:")
    if os.path.exists('Data'):
        print("Fichiers dans Data/:", os.listdir('Data'))
    else:
        print("Le dossier Data/ n'existe pas")
    raise FileNotFoundError("Ajuste le chemin du fichier")

print("\nColonnes:", df.columns.tolist())
print("\nAperçu:\n", df.head())

✅ Fichier chargé: Non supervisé/Data/logs cloud.csv
Shape initial: (1939207, 18)

Colonnes: ['eventID', 'eventTime', 'sourceIPAddress', 'userAgent', 'eventName', 'eventSource', 'awsRegion', 'eventVersion', 'userIdentitytype', 'eventType', 'userIdentityaccountId', 'userIdentityprincipalId', 'userIdentityarn', 'userIdentityaccessKeyId', 'userIdentityuserName', 'errorCode', 'errorMessage', 'requestParametersinstanceType']

Aperçu:
                                 eventID eventTime sourceIPAddress  \
0  3038ebd2-c98a-4c65-9b6e-e22506292313   2017-02         255.253   
1  22a0d9b1-deea-4d39-827b-2af7050ed3f3   2017-02         255.253   
2  9facf7ca-cb76-4b19-940c-3de6803f7efb   2017-02         255.253   
3  6596d3b4-7c98-40b1-867d-f317f1dbdc18   2017-02         255.253   
4  9f9d038c-e5a5-443e-83d5-4cf00941d399   2017-02         255.253   

               userAgent                 eventName        eventSource  \
0             [S3Console               ListBuckets   s3.amazonaws.com   
1  con

## 2. Nettoyage et feature engineering
Extraction des features temporelles, gestion des erreurs AWS.

In [2]:

# 1. Supprimer les colonnes 100% vides ou inutiles
df_clean = df.dropna(axis=1, how='all')
cols_to_drop = ['requestParametersinstanceType']  # colonne vide
df_clean = df_clean.drop(columns=[c for c in cols_to_drop if c in df_clean.columns])

# 2. Gérer les valeurs manquantes
df_clean['errorMessage'] = df_clean['errorMessage'].fillna('NoError')
df_clean['errorCode'] = df_clean['errorCode'].fillna('NoError')

# 3. Nettoyer les types
df_clean['eventTime'] = pd.to_datetime(df_clean['eventTime'], errors='coerce')
df_clean['userIdentityaccountId'] = df_clean['userIdentityaccountId'].astype(str).str.replace('.0', '', regex=False)

# 4. Extraire des features temporelles
df_clean['hour'] = df_clean['eventTime'].dt.hour
df_clean['day_of_week'] = df_clean['eventTime'].dt.dayofweek
df_clean['month'] = df_clean['eventTime'].dt.month

# 5. Créer une colonne "est_erreur" (signal utile même en non supervisé)
df_clean['is_error'] = (df_clean['errorCode'] != 'NoError').astype(int)

# 6. Nettoyer les IP (remplacer valeurs aberrantes)
df_clean['sourceIPAddress'] = df_clean['sourceIPAddress'].replace('255.253', 'unknown')

# 7. Supprimer les lignes avec des dates NaN (si trop peu)
df_clean = df_clean.dropna(subset=['eventTime'])

print("\n✅ Shape après nettoyage:", df_clean.shape)
print("\nColonnes disponibles:", df_clean.columns.tolist())
print("\nAperçu:\n", df_clean.head())

# 1. Supprimer les colonnes 100% vides ou inutiles
df_clean = df.dropna(axis=1, how='all')
cols_to_drop = ['requestParametersinstanceType']  # colonne vide
df_clean = df_clean.drop(columns=[c for c in cols_to_drop if c in df_clean.columns])

# 2. Gérer les valeurs manquantes
df_clean['errorMessage'] = df_clean['errorMessage'].fillna('NoError')
df_clean['errorCode'] = df_clean['errorCode'].fillna('NoError')

# 3. Nettoyer les types
df_clean['eventTime'] = pd.to_datetime(df_clean['eventTime'], errors='coerce')
df_clean['userIdentityaccountId'] = df_clean['userIdentityaccountId'].astype(str).str.replace('.0', '', regex=False)

# 4. Extraire des features temporelles
df_clean['hour'] = df_clean['eventTime'].dt.hour
df_clean['day_of_week'] = df_clean['eventTime'].dt.dayofweek
df_clean['month'] = df_clean['eventTime'].dt.month

# 5. Créer une colonne "est_erreur"
df_clean['is_error'] = (df_clean['errorCode'] != 'NoError').astype(int)

# 6. Nettoyer les IP (remplacer valeurs aberrantes)
df_clean['sourceIPAddress'] = df_clean['sourceIPAddress'].replace('255.253', 'unknown')

# 7. Supprimer les lignes avec des dates NaN
df_clean = df_clean.dropna(subset=['eventTime'])

# ── NOUVEAU — Nettoyer les IPs mal formatées ──────────────────
import re

def is_valid_ip(ip):
    pattern = r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$'
    return bool(re.match(pattern, str(ip)))

df_clean['sourceIPAddress'] = df_clean['sourceIPAddress'].apply(
    lambda x: x if is_valid_ip(x) else "unknown"
)

print(f"✅ IPs valides   : {df_clean['sourceIPAddress'].ne('unknown').sum()}")
print(f"✅ IPs invalides : {df_clean['sourceIPAddress'].eq('unknown').sum()}")
print(f"\nExemples IPs valides :")
print(df_clean[df_clean['sourceIPAddress'] != 'unknown']['sourceIPAddress'].head(5).values)

# ─────────────────────────────────────────────────────────────
print("\n✅ Shape après nettoyage:", df_clean.shape)
print("\nColonnes disponibles:", df_clean.columns.tolist())
print("\nAperçu:\n", df_clean.head())


✅ Shape après nettoyage: (1939207, 21)

Colonnes disponibles: ['eventID', 'eventTime', 'sourceIPAddress', 'userAgent', 'eventName', 'eventSource', 'awsRegion', 'eventVersion', 'userIdentitytype', 'eventType', 'userIdentityaccountId', 'userIdentityprincipalId', 'userIdentityarn', 'userIdentityaccessKeyId', 'userIdentityuserName', 'errorCode', 'errorMessage', 'hour', 'day_of_week', 'month', 'is_error']

Aperçu:
                                 eventID  eventTime sourceIPAddress  \
0  3038ebd2-c98a-4c65-9b6e-e22506292313 2017-02-01         unknown   
1  22a0d9b1-deea-4d39-827b-2af7050ed3f3 2017-02-01         unknown   
2  9facf7ca-cb76-4b19-940c-3de6803f7efb 2017-02-01         unknown   
3  6596d3b4-7c98-40b1-867d-f317f1dbdc18 2017-02-01         unknown   
4  9f9d038c-e5a5-443e-83d5-4cf00941d399 2017-02-01         unknown   

               userAgent                 eventName        eventSource  \
0             [S3Console               ListBuckets   s3.amazonaws.com   
1  console.amazona

## 3. Encodage et normalisation

In [3]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pandas as pd

# ========== PRÉPARATION DES FEATURES ==========

# Colonnes à encoder
categorical_cols = ['eventName', 'eventSource', 'userAgent', 'errorCode', 
                    'sourceIPAddress', 'userIdentitytype', 'awsRegion']

# Colonnes numériques
numerical_cols = ['hour', 'day_of_week', 'month', 'is_error']

# ========== ENCODAGE SIMPLE (sans category_encoders) ==========

df_encoded = df_clean.copy()

for col in categorical_cols:
    if col in df_encoded.columns:
        le = LabelEncoder()
        df_encoded[col + '_encoded'] = le.fit_transform(df_encoded[col].astype(str))
        df_encoded = df_encoded.drop(columns=[col])

# ========== SCALING ==========

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_encoded[numerical_cols + 
                                           [c for c in df_encoded.columns if '_encoded' in c]])

print("✅ Shape finale:", X_scaled.shape)

✅ Shape finale: (1939207, 11)


## 4. Modèle — Isolation Forest
Détection d'anomalies dans les logs AWS CloudTrail.

In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib

# ========== PRÉPARATION ==========
df_encoded = df_clean.copy()

for col in categorical_cols:
    if col in df_encoded.columns:
        le = LabelEncoder()
        df_encoded[col + '_encoded'] = le.fit_transform(df_encoded[col].astype(str))
        df_encoded = df_encoded.drop(columns=[col])

numerical_cols = ['hour', 'day_of_week', 'month', 'is_error']
feature_cols = numerical_cols + [c for c in df_encoded.columns if '_encoded' in c]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_encoded[feature_cols])

# ========== MODÈLE ISOLATION FOREST ==========
model = IsolationForest(contamination=0.1, random_state=42)
predictions = model.fit_predict(X_scaled)

# -1 = anomalie, 1 = normal
anomalies = predictions == -1

# ========== SAUVEGARDE ==========
# Sauvegarder le modèle
joblib.dump(model, 'isolation_forest_model.pkl')
print("✅ Modèle sauvegardé: isolation_forest_model.pkl")

# Sauvegarder les prédictions
results_df = pd.DataFrame({
    'prediction': predictions,  # -1 ou 1
    'is_anomaly': anomalies
})
results_df.to_csv('anomaly_detection_results.csv', index=False)
print("✅ Résultats sauvegardés: anomaly_detection_results.csv")

# ========== STATISTIQUES ==========
print(f"\n📊 Résultats:")
print(f"  - Normal: {(predictions == 1).sum()} / {len(predictions)} ({100*(predictions == 1).sum()/len(predictions):.1f}%)")
print(f"  - Anomalies: {(predictions == -1).sum()} / {len(predictions)} ({100*(predictions == -1).sum()/len(predictions):.1f}%)")

✅ Modèle sauvegardé: isolation_forest_model.pkl
✅ Résultats sauvegardés: anomaly_detection_results.csv

📊 Résultats:
  - Normal: 1745368 / 1939207 (90.0%)
  - Anomalies: 193839 / 1939207 (10.0%)


## 5. Résultats

In [5]:
import pandas as pd
pd.DataFrame({
    'prediction': predictions,
    'is_anomaly': anomalies
}).head(10)

,prediction,is_anomaly
0,-1,True
1,-1,True
2,-1,True
3,-1,True
4,-1,True
5,-1,True
6,-1,True
7,-1,True
8,-1,True
9,-1,True


## 6. Export — output normalisé pour la corrélation

In [6]:
import pandas as pd, os

base = r"C:\Users\asus\Desktop\IDS\Non supervisé\Notebooks\outputs"
os.makedirs(base, exist_ok=True)

df_cloud_normalized = pd.DataFrame({
    "timestamp"   : df_clean["eventTime"].astype(str).values
                    if "eventTime" in df_clean.columns else "unknown",
    "source_ip"   : df_clean["sourceIPAddress"].values
                    if "sourceIPAddress" in df_clean.columns else "unknown",
    "dest_ip"     : "unknown",
    "log_source"  : "cloud",
    "user_id"     : df_clean["userIdentityaccountId"].astype(str).values
                    if "userIdentityaccountId" in df_clean.columns else "unknown",
    "event_name"  : df_clean["eventName"].values
                    if "eventName" in df_clean.columns else "unknown",
    "anomaly_score" : model.decision_function(X_scaled),
    "prediction"  : predictions,
    "alert_level" : ["HIGH" if p == -1 else "NORMAL" for p in predictions],
})

df_cloud_normalized.to_csv(
    os.path.join(base, "cloud_output_normalized.csv"), index=False
)
print("✅ cloud_output_normalized.csv sauvegardé")
print(f"   {len(df_cloud_normalized)} lignes")
print(df_cloud_normalized.head())

✅ cloud_output_normalized.csv sauvegardé
   1939207 lignes
    timestamp source_ip  dest_ip log_source       user_id  \
0  2017-02-01   unknown  unknown      cloud  811596193553   
1  2017-02-01   unknown  unknown      cloud  811596193553   
2  2017-02-01   unknown  unknown      cloud  811596193553   
3  2017-02-01   unknown  unknown      cloud  811596193553   
4  2017-02-01   unknown  unknown      cloud  811596193553   

                 event_name  anomaly_score  prediction alert_level  
0               ListBuckets      -0.009247          -1        HIGH  
1  GetAccountPasswordPolicy      -0.060638          -1        HIGH  
2         GetAccountSummary      -0.005092          -1        HIGH  
3        ListAccountAliases      -0.013849          -1        HIGH  
4            ListMFADevices      -0.028267          -1        HIGH  


## 7. Sauvegarde du modèle

In [7]:
import joblib, os
base = r"C:\Users\asus\Desktop\IDS\models"
os.makedirs(base, exist_ok=True)
joblib.dump(model,  base + r"\cloud_isolation_forest.pkl")
joblib.dump(scaler, base + r"\cloud_scaler.pkl")
print("✅ cloud model sauvegardé")

✅ cloud model sauvegardé


In [8]:
import joblib, json, os
base = r"C:\Users\asus\Desktop\IDS\models"

# Extraire les noms des features à partir de df_encoded
numerical_cols_list = ['hour', 'day_of_week', 'month', 'is_error']
encoded_cols = [c for c in df_encoded.columns if '_encoded' in c]
feature_names = numerical_cols_list + encoded_cols

with open(base + r"\cloud_features.json", "w") as f:
    json.dump(feature_names, f)
print("✅ cloud features:", feature_names)

✅ cloud features: ['hour', 'day_of_week', 'month', 'is_error', 'eventName_encoded', 'eventSource_encoded', 'userAgent_encoded', 'errorCode_encoded', 'sourceIPAddress_encoded', 'userIdentitytype_encoded', 'awsRegion_encoded']
